# Module 03: Edge Infrastructure Reverse Proxies — Interactive Laboratory

Every cell below runs the module's **real** implementation from
`project_solution/api_gateway_proxy.py`. Nothing here prints a claim it has not verified.

What you will do:

1. Load the engine and inspect what it actually exports.
2. Run its primary workflow and check the assertions that define correctness.
3. **Commit to a prediction**, then run the cell that tests it.
4. Measure a property rather than asserting one.
5. Fix a deliberately broken cell in place.

> The code in cells 4, 6 and 8 is lifted from this module's own test suite, so it
> cannot drift from the implementation. If the API changes, those tests fail
> first and this notebook is regenerated from them.


## 1. Load the engine and introspect it

Rather than trusting a hardcoded list of class names, ask the module what it
actually contains.


In [ ]:
import inspect
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve() / 'project_solution'))
import api_gateway_proxy

classes = [n for n, o in inspect.getmembers(api_gateway_proxy, inspect.isclass)
           if o.__module__ == 'api_gateway_proxy']
functions = [n for n, o in inspect.getmembers(api_gateway_proxy, inspect.isfunction)
             if o.__module__ == 'api_gateway_proxy']

print('module   : api_gateway_proxy')
print(f'classes  : {classes}')
print(f'functions: {functions}')
print()
for name in classes:
    obj = getattr(api_gateway_proxy, name)
    try:
        sig = inspect.signature(obj.__init__)
        params = [p for p in sig.parameters if p != 'self']
    except (TypeError, ValueError):
        params = ['<builtin>']
    print(f'  {name}({", ".join(params)})')

## 2. Baseline: Token bucket rate limiter

This is the module's own `test_token_bucket_rate_limiter` — real instantiation, real calls, real
assertions. If it runs clean, the property it encodes holds.


In [ ]:
from api_gateway_proxy import APIGateway, ServiceCluster, TokenBucketRateLimiter, UpstreamServer

limiter = TokenBucketRateLimiter(capacity=3, refill_rate_per_sec=10.0)

# 3 bursts allowed
assert await limiter.allow_request() is True
assert await limiter.allow_request() is True
assert await limiter.allow_request() is True

# 4th burst rejected
assert await limiter.allow_request() is False

print('PASSED: test_token_bucket_rate_limiter')

## 3. 🔮 Prediction — commit before you run

A reverse proxy sits in front of 3 origins. If one origin starts returning 500s but keeps accepting TCP connections, will a naive TCP health probe notice? Write down yes or no before running.

Write your answer down. An uncommitted guess teaches nothing, because you will
retro-fit it to whatever the next cell prints.

The next cell runs `test_path_routing_and_round_robin`, which tests exactly this property.


In [ ]:
async def gateway_env() -> APIGateway:
    gateway = APIGateway(default_rate_limit_capacity=5, default_refill_per_sec=2.0)

    # 1. User Cluster with 2 servers
    async def user_handler_1(req: dict) -> dict:
        return {"service": "user", "node": "user-1", "user_id": req.get("id")}

    async def user_handler_2(req: dict) -> dict:
        return {"service": "user", "node": "user-2", "user_id": req.get("id")}

    user_cluster = ServiceCluster(
        name="user-service",
        servers=[
            UpstreamServer(server_id="usr-1", url="http://10.0.1.1:8080", handler=user_handler_1),
            UpstreamServer(server_id="usr-2", url="http://10.0.1.2:8080", handler=user_handler_2),
        ],
    )

    # 2. Orders Cluster with 1 flaky server
    async def flaky_handler(req: dict) -> dict:
        raise ConnectionResetError("Connection refused by upstream db")

    order_cluster = ServiceCluster(
        name="order-service",
        servers=[
            UpstreamServer(server_id="ord-1", url="http://10.0.2.1:8080", handler=flaky_handler),
        ],
    )

    gateway.register_cluster("/api/v1/users", user_cluster)
    gateway.register_cluster("/api/v1/orders", order_cluster)

    return gateway

_make_gateway_env = gateway_env

gateway_env = await _make_gateway_env()

resp1 = await gateway_env.route_request("/api/v1/users/123", client_ip="192.168.1.10", payload={"id": 123})
assert resp1["status_code"] == 200
assert resp1["upstream"] == "usr-1"
assert resp1["response"]["node"] == "user-1"
assert "trace_id" in resp1

# Request 2 -> usr-2 (Round Robin)
resp2 = await gateway_env.route_request("/api/v1/users/124", client_ip="192.168.1.10", payload={"id": 124})
assert resp2["status_code"] == 200
assert resp2["upstream"] == "usr-2"
assert resp2["response"]["node"] == "user-2"

# Request 3 -> usr-1 again
resp3 = await gateway_env.route_request("/api/v1/users/125", client_ip="192.168.1.10", payload={"id": 125})
assert resp3["status_code"] == 200
assert resp3["upstream"] == "usr-1"

print('PASSED: test_path_routing_and_round_robin')

## 4. Measure it: Unmatched route returns 404

An assertion tells you a property holds. A measurement tells you *how much*.
This cell runs `test_unmatched_route_returns_404` and times it.


In [ ]:
import time

_t0 = time.perf_counter()

async def gateway_env() -> APIGateway:
    gateway = APIGateway(default_rate_limit_capacity=5, default_refill_per_sec=2.0)

    # 1. User Cluster with 2 servers
    async def user_handler_1(req: dict) -> dict:
        return {"service": "user", "node": "user-1", "user_id": req.get("id")}

    async def user_handler_2(req: dict) -> dict:
        return {"service": "user", "node": "user-2", "user_id": req.get("id")}

    user_cluster = ServiceCluster(
        name="user-service",
        servers=[
            UpstreamServer(server_id="usr-1", url="http://10.0.1.1:8080", handler=user_handler_1),
            UpstreamServer(server_id="usr-2", url="http://10.0.1.2:8080", handler=user_handler_2),
        ],
    )

    # 2. Orders Cluster with 1 flaky server
    async def flaky_handler(req: dict) -> dict:
        raise ConnectionResetError("Connection refused by upstream db")

    order_cluster = ServiceCluster(
        name="order-service",
        servers=[
            UpstreamServer(server_id="ord-1", url="http://10.0.2.1:8080", handler=flaky_handler),
        ],
    )

    gateway.register_cluster("/api/v1/users", user_cluster)
    gateway.register_cluster("/api/v1/orders", order_cluster)

    return gateway

_make_gateway_env = gateway_env

gateway_env = await _make_gateway_env()

resp = await gateway_env.route_request("/api/v1/unknown", client_ip="192.168.1.10")
assert resp["status_code"] == 404
assert "No upstream cluster found" in resp["error"]

_elapsed = (time.perf_counter() - _t0) * 1000
print('PASSED: test_unmatched_route_returns_404')
print(f'wall clock: {_elapsed:.2f} ms')

## 5. 🛠️ Fix this cell — it is deliberately broken

The cell below asserts something **false** about the real object. Read the
failure, work out the true value from the module's actual behaviour, and correct
the expected number.

Do not delete the assertion. The point is to make it pass by knowing the answer.


In [ ]:
# DELIBERATELY BROKEN - fix the expected value below.
# Hint: print the real value first, then decide what the assertion should say.

exports = [n for n in dir(api_gateway_proxy) if not n.startswith('_')]
print(f'actual export count: {len(exports)}')
print(f'actual exports     : {exports}')

EXPECTED_EXPORT_COUNT = 999      # <-- wrong on purpose. Replace it.

assert len(exports) == EXPECTED_EXPORT_COUNT, (
    f'expected {EXPECTED_EXPORT_COUNT} exports, found {len(exports)}. '
    'Read the printed value above and correct the constant.'
)
print('Fixed - assertion now reflects reality.')

### 🎓 Key takeaways

1. A health probe that only checks TCP tells you the process is alive, not correct.
2. The edge is where you terminate TLS, cache, rate-limit and shed load.
3. Every hop you add is a hop that can fail independently.

---

**Continue with this module:**

- [README.md](README.md) — the mental model and failure modes
- [PROJECT_GUIDE.md](PROJECT_GUIDE.md) — build it yourself, in 3 tiers
- [starter/](starter/) — your stubs; run the tests from there to grade yourself
- [debug_lab/SYMPTOMS.md](debug_lab/SYMPTOMS.md) — diagnose planted bugs from the symptom
- [TROUBLESHOOTING_AND_EDGE_CASES.md](TROUBLESHOOTING_AND_EDGE_CASES.md) — real errors, real causes
- [SELF_ASSESSMENT_AND_CHALLENGES.md](SELF_ASSESSMENT_AND_CHALLENGES.md) — quiz and diagnostics
